In [1]:
import pandas as pd
import PyPDF2
import os
import seaborn as sns
import openai
import matplotlib.pyplot as plt
import numpy as np
import re
from textstat import flesch_reading_ease
import pandas as pd
import spacy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time
import pandas as pd
import stata_setup
import matplotlib.ticker as mticker


In [18]:
import linearmodels
print(linearmodels.__version__) 

4.25


In [44]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

In [2]:
### DataFrame and Graphs Settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colum', None)
pd.set_option("display.max_colwidth", None)

sns.set(font_scale=1.5)
sns.set(style='white', font_scale=1.5)

In [63]:
df= pd.read_parquet('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/compustat_database.parquet')


In [64]:
df.Year.unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
       2021])

In [65]:
df = df.set_index(["FIPS", "Year"]) 

In [75]:
del df

## Baseline

In [66]:
X = df["bartik"]
y = df["main_jobs"]
model = PanelOLS(y, X, time_effects=True)
results = model.fit(cov_type="clustered", cluster_entity=True)

print(results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              main_jobs   R-squared:                     6.135e-06
Estimator:                   PanelOLS   R-squared (Between):          -7.223e-05
No. Observations:            10159031   R-squared (Within):            1.014e-05
Date:                Tue, Feb 11 2025   R-squared (Overall):          -4.327e-05
Time:                        14:26:15   Log-likelihood                -6.643e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      62.329
Entities:                        3175   P-value                           0.0000
Avg Obs:                       3199.7   Distribution:              F(1,10159018)
Min Obs:                       1.0000                                           
Max Obs:                    1.975e+05   F-statistic (robust):             9.6378
                            

In [67]:
df.dtypes

gvkey                                       object
tic                                         object
cusip                                       object
comCounty                                   object
comDom                                    category
comCountry                                  object
comNAICS                                     int64
comHeadquarters                             object
OccFam                                       int64
OccFamName                                category
SOC                                         object
SOCName                                     object
CanonEmployer                               object
Sector                                       int64
SectorName                                category
NAICS3                                       int64
NAICS4                                       int64
NAICS5                                       int64
NAICS6                                       int64
State                          

## FIPS/MSA controls

In [69]:
X = df[["bartik", "percapUIC", "percapInc","retirement", "lpctunion", "black_msa", "intmig", "dommig","college_st"]]
y = df["main_jobs"]

model = PanelOLS(y, X, entity_effects=True,  time_effects=True)
results = model.fit(cov_type="clustered", cluster_entity=True)
print(results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              main_jobs   R-squared:                     3.255e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.1942
No. Observations:             8581524   R-squared (Within):              -0.0024
Date:                Tue, Feb 11 2025   R-squared (Overall):             -0.1572
Time:                        14:32:09   Log-likelihood                -5.558e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      31.032
Entities:                        1729   P-value                           0.0000
Avg Obs:                       4963.3   Distribution:               F(9,8579775)
Min Obs:                       1.0000                                           
Max Obs:                    1.965e+05   F-statistic (robust):             2.0330
                            

## Job Controls

In [74]:
X = df[["bartik", "percapUIC", "percapInc","retirement", "lpctunion", "black_msa", "intmig", "dommig","college_st", 'Degree', 'boncom', 'length']]
y = df["main_jobs"]

model = PanelOLS(y, X, entity_effects=True, time_effects=True,other_effects=df[["Sector", "OccFam"]])
results = model.fit(cov_type="clustered", cluster_entity=True)
print(results.summary)

ValueError: At most two effects supported.